In [16]:
from statsbombpy import sb
import pandas as pd
import xgboost as xgb
print("imports done")

imports done


In [28]:
if os.path.exists('../data/shots_raw.csv'):
    shots_df = pd.read_csv('../data/shots_raw.csv')
    print(f"Loaded from cache: {len(shots_df)} shots")
else:
    print("No cache found - run the extraction loop")

Loaded from cache: 21153 shots


In [29]:
import ast
def parse_location(loc):
    if isinstance(loc, list):
        return loc
    if isinstance(loc, str):
        import ast
        return ast.literal_eval(loc)
    return None

shots_df['location'] = shots_df['location'].apply(parse_location)
shots_df = shots_df.copy()
print("Location column fixed")
print("Sample:", shots_df['location'].iloc[0])

Location column fixed
Sample: [108.6, 28.0]


In [18]:
from statsbombpy import sb
comps = sb.competitions()
print(comps.shape)
print(comps[['competition_name', 'season_name']].to_string())

(80, 12)
           competition_name season_name
0             1. Bundesliga   2023/2024
1             1. Bundesliga   2015/2016
2    African Cup of Nations        2023
3          Champions League   2018/2019
4          Champions League   2017/2018
5          Champions League   2016/2017
6          Champions League   2015/2016
7          Champions League   2014/2015
8          Champions League   2013/2014
9          Champions League   2012/2013
10         Champions League   2011/2012
11         Champions League   2010/2011
12         Champions League   2009/2010
13         Champions League   2008/2009
14         Champions League   2006/2007
15         Champions League   2004/2005
16         Champions League   2003/2004
17         Champions League   1999/2000
18         Champions League   1972/1973
19         Champions League   1971/1972
20         Champions League   1970/1971
21             Copa America        2024
22             Copa del Rey   1983/1984
23             Copa del Rey   1

In [19]:
la_liga = comps[comps['competition_name'] == 'La Liga']
print(la_liga[['competition_id', 'season_id', 'competition_name', 'season_name']].to_string())

    competition_id  season_id competition_name season_name
40              11         90          La Liga   2020/2021
41              11         42          La Liga   2019/2020
42              11          4          La Liga   2018/2019
43              11          1          La Liga   2017/2018
44              11          2          La Liga   2016/2017
45              11         27          La Liga   2015/2016
46              11         26          La Liga   2014/2015
47              11         25          La Liga   2013/2014
48              11         24          La Liga   2012/2013
49              11         23          La Liga   2011/2012
50              11         22          La Liga   2010/2011
51              11         21          La Liga   2009/2010
52              11         41          La Liga   2008/2009
53              11         40          La Liga   2007/2008
54              11         39          La Liga   2006/2007
55              11         38          La Liga   2005/20

## Pull match IDs across multiple La Liga seasons

What and why: StatsBomb data is structured as competition → matches → events. We can't directly pull all shots — we first need to get the list of match IDs for each season, then loop through matches to extract shot events. We'll pull matches for several recent seasons to get a large enough shot dataset to train on.

We'll skip 1973/1974 and use 2004/2005 through 2020/2021 — that's 16 seasons of La Liga, potentially 50,000+ shots which is plenty for gradient boosting.

In [20]:
import pandas as pd

season_ids = [90, 42, 4, 1, 2, 27, 26, 25, 24, 23, 22, 21, 41, 40, 39, 38, 37]

all_matches = [] #Contains a list of all matches dataframes

for id in season_ids:
    matches = sb.matches(competition_id = '11', season_id = id) 
    all_matches.append(matches) 
    print(f"Season {id} : {len(matches)} Matches")

matches_df = pd.concat(all_matches, ignore_index = True) # Contains all matches into a single dataframe. 
# Ignore index is to reset the row index numbers so that they go cleanly from 0 to total-1
print(f"Total matches: {len(matches_df)}")
print(matches_df.columns.tolist())

c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Season 90 : 35 Matches
Season 42 : 33 Matches
Season 4 : 34 Matches


c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Season 1 : 36 Matches


c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Season 2 : 34 Matches
Season 27 : 380 Matches


c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Season 26 : 38 Matches
Season 25 : 31 Matches
Season 24 : 32 Matches
Season 23 : 37 Matches
Season 22 : 33 Matches
Season 21 : 35 Matches
Season 41 : 31 Matches
Season 40 : 27 Matches
Season 39 : 26 Matches
Season 38 : 17 Matches
Season 37 : 7 Matches
Total matches: 866
['match_id', 'match_date', 'kick_off', 'home_score', 'away_score', 'match_status', 'match_status_360', 'last_updated', 'last_updated_360', 'match_week', 'competition_id', 'competition_country_name', 'competition_name', 'competition', 'season_id', 'season', 'home_team_id', 'home_team', 'home_team_gender', 'home_team_group', 'home_team_country_id', 'home_team_country_name', 'away_team_id', 'away_team', 'away_team_gender', 'away_team_group', 'away_team_country_id', 'away_team_country_name', 'competition_stage_id', 'competition_stage', 'stadium_id', 'stadium', 'stadium_country_id', 'stadium_country_name', 'referee_id', 'referee', 'referee_country_id', 'referee_country_name', 'home_managers', 'away_managers', 'home_manager_i

## Extract shot events from all matches

What and why: Each match contains hundreds of events (passes, tackles, shots, etc.). We only want shot events — these become the rows of our training dataset, one row per shot. Each shot will eventually have features like distance, angle, body part, and our target variable will be whether it resulted in a goal.

In [21]:
match_ids = matches_df['match_id'].tolist()
all_shots = []

for i, matchid in enumerate(match_ids):
    events = sb.events(match_id = matchid)
    shots = events[events['type'] == 'Shot'].copy()
    all_shots.append(shots)

    if i%50 == 0:
        print(f"Processed {i}/{len(match_ids)} matches, shots so far: {sum(len(s) for s in all_shots)}")

shots_df = pd.concat(all_shots, ignore_index = True)
print(f"\nTotal shots: {len(shots_df)}")
print(f"Columns: {shots_df.columns.tolist()}")

shots_df.to_csv('../data/shots_raw.csv', index=False)
print("Saved shots_raw.csv")

Processed 0/866 matches, shots so far: 29


c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shree

Processed 50/866 matches, shots so far: 1238


c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shree

Processed 100/866 matches, shots so far: 2462


...

Processed 800/866 matches, shots so far: 19497


c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shree

Processed 850/866 matches, shots so far: 20804


c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shreesh\projects\world_cup\match_predictor\venv\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\shree\Shree


Total shots: 21153
Columns: ['50_50', 'bad_behaviour_card', 'ball_receipt_outcome', 'ball_recovery_offensive', 'ball_recovery_recovery_failure', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_offensive', 'foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'interception_outcome', 'location', 'match_id', 'minute', 'miscontrol_aerial_won', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_height', 'pass_inswinging', 'pass_length', 'pass_no

## Explore and clean the shot data

What and why: Before building features we need to understand what the raw data looks like — check for nulls, understand the target variable distribution, and extract x/y coordinates from the location column (it's currently stored as a list [x, y], not two separate columns)

In [22]:
print(shots_df['shot_outcome'].value_counts())
print()
print(shots_df['shot_body_part'].value_counts())
print()
print(shots_df['shot_type'].value_counts())
print()
print(f"Location Sample: {shots_df['location'].iloc[0]}")

shot_outcome
Off T               6708
Saved               5343
Blocked             4804
Goal                2648
Wayward              974
Post                 528
Saved Off Target      87
Saved to Post         61
Name: count, dtype: int64

shot_body_part
Right Foot    10602
Left Foot      7485
Head           3017
Other            49
Name: count, dtype: int64

shot_type
Open Play    19584
Free Kick     1322
Penalty        245
Corner           2
Name: count, dtype: int64

Location Sample: [108.6, 28.0]


## Feature engineering

What and why: The raw data has useful information but not in model-ready form. We need to extract x/y from location, compute distance and angle to goal mathematically, and encode categorical columns like shot_body_part and shot_type as numbers

In [ ]:
import numpy as np

# Extract x, y coordinates
shots_df['x'] = shots_df['location'].apply(lambda loc: loc[0])
shots_df['y'] = shots_df['location'].apply(lambda loc: loc[1])

# StatsBomb pitch: 120x80 units, goal at x=120, y=40 (center)
GOAL_X = 120
GOAL_Y = 40

# Distance to goal (straight line)
shots_df['distance'] = np.sqrt((shots_df['x']-GOAL_X)**2+(shots_df['y']-GOAL_Y)**2)

# Angle to goal in degrees
# atan2 gives the angle between the shot location and the goal center
shots_df['angle'] = np.degrees(np.arctan2(np.abs(shots_df['y']-GOAL_Y), np.abs(shots_df['x']-GOAL_X)))

print(shots_df[['x', 'y', 'distance', 'angle']].describe().round(2))

              x         y  distance     angle
count  21153.00  21153.00  21153.00  21153.00
mean     103.78     39.48     19.11     28.58
std        8.75     10.05      8.69     19.07
min       40.00      0.70      0.40      0.00
25%       97.70     31.80     12.16     12.48
50%      105.30     39.60     18.18     26.83
75%      110.60     47.00     25.23     42.19
max      120.00     78.80     81.44     90.00
